In [ ]:
import pypsa
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import plotly.express as px
import hvplot.pandas
import matplotlib.colors as mcolors

In [ ]:
scenario_name = "nz_2035_exp"  # scenario name, default value is "" for tutorial or default configuration
                    # value shall be non null if a scenario name is specified under the "run" tag in the config file

scenario_subpath = scenario_name + "/" if scenario_name else ""

export_level = 0

n=pypsa.Network(f"../../results/" + scenario_name + f"/postnetworks/elec_s_10_ec_lc3.0_Co2L_3H_2035_0.071_AB_{export_level}export.nc")
regions_onshore = gpd.read_file("../../results/shapes/country_shapes.geojson")

### Get statistics

In [ ]:
n.statistics.dispatch(comps=["Generator", "StorageUnit"]).div(1e6).round(1) # "StorageUnit",

### Get dispatch

In [ ]:
threshold_dispatch = 1e6

dispatch = n.statistics.dispatch(bus_carrier="AC")[n.statistics.dispatch(bus_carrier="AC") > 0]
threshold_dispatch_sum = dispatch[dispatch < threshold_dispatch].sum()
dispatch = dispatch[dispatch > threshold_dispatch]
dispatch = pd.DataFrame(dispatch.droplevel(0)).T
dispatch[f"(Dispatch < {threshold_dispatch/1e6} TWh thres.)"] = threshold_dispatch_sum
dispatch = dispatch.T.div(1e6).round(2)

In [ ]:
dispatch

### RES share

In [ ]:
res_techs = [
    "Geothermal",
    "Offshore Wind (AC)",
    "Offshore Wind (DC)",
    "Onshore Wind",
    "Run of River",
    "Solar",
    "Reservoir & Dam",
]

fossil_techs = [
    "Coal",
    "Combined-Cycle Gas",
    "Open-Cycle Gas",
    "urban central gas CHP",
    "Oil",
]

In [ ]:
re = dispatch[dispatch.index.isin(res_techs)].sum().values[0]

In [ ]:
fossil = dispatch[dispatch.index.isin(fossil_techs)].sum().values[0]

In [ ]:
# Check, if re + fossil = total dispatch - threshold

if abs(dispatch.sum().values[0] - threshold_dispatch_sum/1e6 - re - fossil) > 0.01:
    raise ValueError("Sum of dispatches does not match threshold dispatch sum")

In [ ]:
re_share = re / (re + fossil)

In [ ]:
print(f"The share of renewable dispatch is {re_share:.2%}.")